# GridNet-HD — build per-zone caches on Colab (Phase 1)

สร้าง `points.npz` (voxel 5 ซม., labels 11 class, geometric features, HAG) ของทุก zone แล้วเก็บลง Drive `GridNet/cache/<zone>/`
ไม่ต้องใช้ GPU: รันบน **CPU High-RAM runtime** (Runtime → Change runtime type → CPU, High-RAM; ~5–20 นาที/zone)
⚠️ Standard CPU runtime (12.7 GB) จะโดน OOM-kill กับ zone ที่มี raw points > ~70M (t4z1 t5a3 t6z2 t6z3 t6z4 t6z6) — ขึ้น `[fail]` โดยไม่มี traceback
Notebook ข้าม zone ที่มีอยู่แล้ว → รันซ้ำหลาย session ได้จนครบ 36 zones (train 21 + val 6 + test 9)

**Test 9 zones (สำหรับ leaderboard)** อยู่ท้ายรายการ: LAS ของ test ไม่มี field `ground_truth` → cache ได้ `labels` = 255 ทุกจุดและ `labeled = False`
(ต้องใช้ `gridnet_code.zip` ของ **update35 ขึ้นไป** — โค้ดก่อนหน้านั้นจะอ่าน field `classification` มาเป็น label; cell 2 ตรวจให้)
test zones ใหญ่กว่าโดยเฉลี่ย (รวม 767 ล้านจุดดิบ = เฉลี่ย 85 ล้านจุดต่อโซน เทียบกับ 63 ล้านของ train/val) → ต้องเป็น **High-RAM**;
เวลาที่วัดได้ (14 พ.ย. 2026, High-RAM 8 cores): 9 โซนใช้ 5.2 ชม. **เมื่อติ๊ก `WITH_ASSOC`** (10–72 นาทีต่อโซน); ไม่ติ๊ก (ค่าเริ่มต้น) ยังไม่ได้วัดกับ test zones · ไม่ใช้ GPU จึงแทบไม่เสีย compute units
**ไม่ต้องติ๊ก `WITH_ASSOC`**: ไม่มีขั้นตอนไหนใช้ `assoc.npz` ของ cache (notebook ที่ต้องใช้ association คำนวณเองบน GPU) — ติ๊กแล้วต้องโหลดภาพทั้งโซน คำนวณบน CPU และได้ไฟล์ 5–13 GB ต่อโซนบน Drive

**ก่อนรัน**: บน Mac `bash scripts/pack_code.sh` แล้วอัปโหลด `data/gridnet_code.zip` ไป Drive `GridNet/code/`

In [ ]:
#@title 1. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'   #@param {type:"string"}
OUT = f'{DRIVE}/cache'
TEST_ZONES = ["t1z4","t1z5a","t1z7","t3z1","t3z2","t3z5","t5a2","t6z1","t6z5"]             # test: no labels (leaderboard)
ALL_ZONES = ["t1z5b","t1z8","t3z4","t4z1","t5b1","t5b5",                                   # val
             "t1z6a","t1z6b","t2z5","t3z3","t3z6","t3z7","t5a1","t5a3","t5a4","t5a5","t5b2",
             "t5b3","t5b4","t5b6","t5c1","t5c2","t5c3","t6z2","t6z3","t6z4","t6z6"] + TEST_ZONES   # train, then test
ZONES = ALL_ZONES   # or a subset, e.g. TEST_ZONES
WITH_ASSOC = False  #@param {type:"boolean"}
os.makedirs(OUT, exist_ok=True)
done = [z for z in ZONES if os.path.exists(f'{OUT}/{z}/points.npz')]
print(f"{len(done)}/{len(ZONES)} zones already cached:", done)
print("to do:", [z for z in ZONES if z not in done])


In [ ]:
#@title 2. Install laspy + unpack code
import subprocess, sys, zipfile, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "laspy[lazrs]", "huggingface_hub"], check=True)
os.makedirs('/content/code', exist_ok=True)
zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
print(os.listdir('/content/code'))
assert 'no ground_truth field' in open('/content/code/gridnet/io.py').read(), "gridnet_code.zip on Drive is older than update35: pack and upload it again (the test zones need it)"


In [ ]:
#@title 3. Download → cache → copy to Drive, one zone at a time (resumable)
import subprocess, sys, os, shutil, time
from huggingface_hub import snapshot_download
patterns = lambda z: [f"{z}/*"] if WITH_ASSOC else [f"{z}/lidar/*", f"{z}/pose/*"]
for z in ZONES:
    if os.path.exists(f'{OUT}/{z}/points.npz'):
        print(f"[skip] {z}"); continue
    t0 = time.time()
    snapshot_download("heig-vd-geo/GridNet-HD", repo_type="dataset", allow_patterns=patterns(z), local_dir="/content/data")
    print(f"[{z}] downloaded in {(time.time() - t0) / 60:.1f} min", flush=True)
    cmd = [sys.executable, "/content/code/scripts/build_cache.py", "--zone", f"/content/data/{z}", "--out", "/content/cache",
           "--voxel", "0.05", "--feat-grid", "0.10", "--k", "8"] + ([] if WITH_ASSOC else ["--no-assoc"])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"[fail] {z}\n", r.stdout[-1500:], r.stderr[-1500:]); continue
    os.makedirs(f'{OUT}/{z}', exist_ok=True)
    for f in os.listdir(f'/content/cache/{z}'):
        shutil.copy(f'/content/cache/{z}/{f}', f'{OUT}/{z}/{f}')
    shutil.rmtree(f'/content/data/{z}', ignore_errors=True); shutil.rmtree(f'/content/cache/{z}', ignore_errors=True)
    print(f"[{z}] done in {(time.time() - t0) / 60:.1f} min  ->  {OUT}/{z}", flush=True)
    print(open(f'{OUT}/{z}/report.txt').read().splitlines()[0])
print("finished", sum(os.path.exists(f'{OUT}/{z}/points.npz') for z in ZONES), "/", len(ZONES))
